In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03_rfm_and_filter - Cálculo RFM con Fecha de Corte
# MAGIC RFM para periodo: 2016-09-04 a 2018-09-30

# COMMAND ----------

from pyspark.sql import functions as F
from datetime import datetime

SILVER_PATH = "/Volumes/olist/olist_silver/silver/"
GOLD_PATH = "/Volumes/olist/olist_gold/gold/"

# Fecha de corte fija (NO usar fecha del sistema)
CUTOFF_DATE = "2018-09-30 23:59:59"
START_DATE = "2016-09-04 21:15:19"

start_time = datetime.now()
print(f"🚀 Inicio: {start_time.strftime('%H:%M:%S')}")
print(f"📅 Periodo RFM: {START_DATE} → {CUTOFF_DATE}\n")

# COMMAND ----------

# Crear estructura Gold
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS olist.olist_gold")
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.gold")
    print("✅ Volume Gold verificado\n")
except:
    pass

# COMMAND ----------

# Cargar orders_full
print("📥 Cargando orders_full...\n")

orders_full = spark.read.format("delta").load(f"{SILVER_PATH}orders_full/")

print(f"✅ Total registros: {orders_full.count():,}\n")

# COMMAND ----------

# Filtrar periodo RFM
print("🔍 Filtrando periodo...\n")

orders_rfm = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(CUTOFF_DATE)) &
    (F.col("order_status") != "canceled")
)

print(f"✅ Registros en periodo: {orders_rfm.count():,}\n")

# COMMAND ----------

# Calcular RFM por customer_id
print("📊 Calculando RFM...\n")

rfm = orders_rfm.groupBy("customer_id").agg(
    # Recency: días desde última compra hasta fecha de corte
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    
    # Frequency: cantidad de pedidos
    F.count("order_id").alias("frequency"),
    
    # Monetary: suma total gastada
    F.sum("payment_sum").alias("monetary")
).filter(
    F.col("customer_id").isNotNull()
)

print(f"✅ RFM calculado para {rfm.count():,} clientes\n")

# COMMAND ----------

# Resumen estadístico
print("📈 Estadísticas RFM:\n")

stats = rfm.select(
    F.min("recency").alias("min_R"),
    F.max("recency").alias("max_R"),
    F.avg("recency").alias("avg_R"),
    F.min("frequency").alias("min_F"),
    F.max("frequency").alias("max_F"),
    F.avg("frequency").alias("avg_F"),
    F.min("monetary").alias("min_M"),
    F.max("monetary").alias("max_M"),
    F.avg("monetary").alias("avg_M")
).collect()[0]

print(f"Recency:   min={stats['min_R']}, max={stats['max_R']}, avg={stats['avg_R']:.1f} días")
print(f"Frequency: min={stats['min_F']}, max={stats['max_F']}, avg={stats['avg_F']:.2f} pedidos")
print(f"Monetary:  min={stats['min_M']:.2f}, max={stats['max_M']:.2f}, avg={stats['avg_M']:.2f}\n")

# COMMAND ----------

# Guardar en Gold
print("💾 Guardando en Gold...\n")

rfm.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}rfm_cutoff_20180930/")

print(f"✅ Guardado: {GOLD_PATH}rfm_cutoff_20180930/")

# COMMAND ----------

# Resumen final
duration = (datetime.now() - start_time).total_seconds()

print(f"\n{'='*60}")
print("✅ PROCESO COMPLETADO")
print(f"{'='*60}")
print(f"📊 Clientes con RFM: {rfm.count():,}")
print(f"📅 Fecha de corte: {CUTOFF_DATE}")
print(f"⏱️  Duración: {duration:.2f} seg")